# Free Colab Multi-Agent Book Writer

Runs **Qwen/Qwen2.5-7B-Instruct** on a Colab GPU. No OpenAI API key or paid inference service is used. The five agent roles share a locally loaded model and keep their separate instructions and review stages.

Before starting: **Runtime → Change runtime type → GPU** (T4 if available). Free GPUs and session lengths are not guaranteed. This notebook has not been GPU-tested in the preparation environment; run it and review the result before submitting. The model download is several GB. Source retrieval still uses the public internet.

Run cells in order. Upload **UPI_Book_Writer_Free_Colab.zip**, the updated ZIP supplied with this notebook, not the earlier API-only version.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess, json

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert len(zip_names) == 1, 'Upload exactly one updated project ZIP.'
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall('/content')
project = Path('/content/upi-book-writer-free-colab')
assert (project / 'bookwriter/local.py').exists(), 'Wrong ZIP: upload UPI_Book_Writer_Free_Colab.zip.'
os.chdir(project)
print('Project ready:', project)


## Install the local-model dependencies
The ordinary demo/tests use only Python's standard library. Only local inference needs these packages.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
assert result.returncode == 0, 'Dependency installation failed; inspect the output above.'

import torch
assert torch.cuda.is_available(), 'No GPU: select Runtime > Change runtime type > GPU and reconnect.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## Run the tests
These test validators and orchestration; they do not certify real-model quality.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])
assert result.returncode == 0, 'Tests failed; inspect the output above.'


## Check public sources before downloading the model
The system requires readable public evidence. Blocked or unavailable sources are reported explicitly; there is no automatic switch to prewritten demo sources.

In [ ]:
from bookwriter.core import Pipeline, SourceFetcher, BASE
cards = json.loads((BASE / 'examples/source_cards.json').read_text())
seeds = [{key: card[key] for key in ['id', 'publisher', 'title', 'url']} for card in cards]
fetcher = SourceFetcher()
readable = []
for seed in seeds:
    try:
        page = fetcher.fetch(seed)
        readable.append(seed)
        print(f"OK [{seed['id']}] {seed['title']} ({len(page['text'])} characters)")
    except Exception as error:
        print(f"UNAVAILABLE [{seed['id']}]: {type(error).__name__}: {error}")
assert len(readable) >= 2, 'Fewer than two sources are readable. Fix source access before generation.'


## Load the free model
This downloads Qwen and loads it in 4-bit precision to reduce GPU memory. No Hugging Face token is required for this public model. A missing-token notice can be harmless; a real download error must be resolved.

In [ ]:
from bookwriter.local import LocalClient
client = LocalClient('Qwen/Qwen2.5-7B-Instruct')
print('Model loaded. Proceed to the next cell.')


## Run the actual five-agent workflow
This generates new chapters with the model, not prewritten demo fixtures. Keep this output visible for your screenshots. If a check fails, the run remains failed and you must inspect the issue; success is not guaranteed.

In [ ]:
from datetime import datetime
from uuid import uuid4

run_dir = project / 'runs' / (datetime.now().strftime('local-%Y%m%d-%H%M%S') + '-' + uuid4().hex[:4])
try:
    pipeline = Pipeline(client, fetcher, seeds, run_dir, max_revisions=2, mode='local')
    status = pipeline.run()
    print('FINAL STATUS:', status['status'])
    print('WORDS:', status.get('words'))
except Exception as error:
    print('RUN FAILED:', type(error).__name__, str(error))
    print('Diagnostics directory:', run_dir)
    print('Continue to the next cell to inspect saved status/review details.')


## Read the result or exact error
This cell makes no model calls.

In [ ]:
status_path = run_dir / 'status.json'
if status_path.exists():
    status = json.loads(status_path.read_text())
    print(json.dumps(status, indent=2))
    reviews = sorted(run_dir.glob('chapter-*-review-*.json'))
    if status.get('status') != 'accepted' and reviews:
        latest = json.loads(reviews[-1].read_text())
        print('Last review issues:', latest.get('issues'))
else:
    print('No saved status. Review the error in the previous cell.')


## Preview and screenshot
Take screenshots of the run output and the final book. Only an `accepted` local run creates a final book. Inspect the citations, links, word counts and grammar yourself too.

In [ ]:
from IPython.display import HTML, display
if (run_dir / 'run.html').exists():
    display(HTML((run_dir / 'run.html').read_text()))
if (run_dir / 'book.html').exists():
    display(HTML((run_dir / 'book.html').read_text()))
else:
    print('No final book: the run did not pass all checks.')


## Download the submission files
This creates a repository ZIP with the code, diagrams, notebook, final book and audit files. It excludes downloaded model weights and the full scraped source pages. There are no API keys in this workflow. Upload the extracted files to your public GitHub repository, then add your screenshots PDF (under 10 MB).

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

status = json.loads((run_dir / 'status.json').read_text())
assert status['status'] == 'accepted', 'Resolve the failed run before creating submission files.'
output = Path('/content/UPI_GitHub_Submission.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in sorted(project.rglob('*')):
        relative = path.relative_to(project)
        if path.is_file() and not any(part in {'runs', '__pycache__', '.git', '.venv'} for part in relative.parts) and path.name != '.env':
            archive.write(path, str(relative))
    for path in sorted(run_dir.iterdir()):
        if path.is_file() and path.name != 'sources.json':
            archive.write(path, 'submission/' + path.name)
files.download(str(output))


## If you still want to diagnose the earlier API run
This optional cell reads the last failed run's status; it does **not** send another API request. If an API key was pasted into a chat or shared notebook, revoke it from the provider's API-key settings. You do not need a replacement for this local-model notebook.


In [ ]:
# Run this only in the OLD notebook/project where the failed API run happened.
# from pathlib import Path
# import json
# previous = Path('runs/live-20261004-085408/status.json')
# if previous.exists():
#     failure = json.loads(previous.read_text())
#     print('Status:', failure.get('status'))
#     print('Error:', failure.get('error'))
# else:
#     print('No status file; the run may have failed before the pipeline started.')
